In [7]:
!pip install geomloss
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
from geomloss import SamplesLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# trasformazioni 
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

#ResNet-18
def get_feature_extractor():
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    # Rimozione dell'ultimo layer di classificazione (fc)
    model.fc = nn.Identity()
    model.eval()
    model.to(device)
    return model

#funzione per estrarre e normalizzare le feature di un dominio
@torch.no_grad()
def extract_domain_features(data_dir, model, batch_size=128):
    dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    features_list = []
    for images, _ in loader:
        images = images.to(device)
        feats = model(images)
        # normalizzazione L2 
        feats = F.normalize(feats, p=2, dim=-1)
        features_list.append(feats.cpu())
        
    return torch.cat(features_list, dim=0)

# Percorso del dataset PACS
def find_pacs_root():
    candidate_paths = ["./pacs_data", "/kaggle/input", "/content", "."]
    for candidate in candidate_paths:
        if os.path.exists(candidate):
            for root, dirs, _ in os.walk(candidate):
                normalized = {d.lower().replace(" ", "_"): d for d in dirs}
                if "photo" in normalized and ("sketch" in normalized or "cartoon" in normalized):
                    return root
    return "./pacs_data"

data_root = find_pacs_root()
domains = ["photo", "art_painting", "cartoon", "sketch"]
domain_labels = ["Photo", "Art Painting", "Cartoon", "Sketch"]

model = get_feature_extractor()

# estrazione delle feature per ciascun dominio
domain_features = {}
available_subdirs = {d.lower().replace(" ", "_"): d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))}

for dom in domains:
    dir_key = dom.lower().replace(" ", "_")
    folder_name = available_subdirs.get(dir_key, dom)
    dom_path = os.path.join(data_root, folder_name)
    domain_features[dom] = extract_domain_features(dom_path, model)

# definizione della divergenza di Sinkhorn
sinkhorn_loss = SamplesLoss(
    loss="sinkhorn", 
    p=2, 
    blur=np.sqrt(0.05), 
    scaling=0.9
)

#matrice 4x4 delle distanze
n_domains = len(domains)
distance_matrix = np.zeros((n_domains, n_domains))

for i in range(n_domains):
    for j in range(i, n_domains):
        if i == j:
            distance_matrix[i, j] = 0.0
        else:
            x_i = domain_features[domains[i]].to(device)
            x_j = domain_features[domains[j]].to(device)
            
            #distanza di trasporto regolarizzata
            with torch.no_grad():
                val = sinkhorn_loss(x_i, x_j).item()
                dist = np.sqrt(max(0.0, val))
                
            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist

#stampa della matrice
header = f"{'Dominio':<15}" + "".join([f"{d:>15}" for d in domain_labels])
print(header)
print("-" * len(header))
for i, dom_label in enumerate(domain_labels):
    row_str = f"{dom_label:<15}"
    for j in range(n_domains):
        row_str += f"{distance_matrix[i, j]:>15.4f}"
    print(row_str)

#calcolo della distanza media dai tre domini sorgente per ciascun target
sketch_idx = domains.index("sketch")
cartoon_idx = domains.index("cartoon")

sources_for_sketch = [idx for idx in range(n_domains) if idx != sketch_idx]
sources_for_cartoon = [idx for idx in range(n_domains) if idx != cartoon_idx]

mean_dist_sketch = np.mean([distance_matrix[sketch_idx, s] for s in sources_for_sketch])
mean_dist_cartoon = np.mean([distance_matrix[cartoon_idx, s] for s in sources_for_cartoon])

print("\nDistanza media dalle rispettive sorgenti:")
print(f"Cartoon target: {mean_dist_cartoon:.4f}")
print(f"Sketch target:  {mean_dist_sketch:.4f}")

Dominio                  Photo   Art Painting        Cartoon         Sketch
---------------------------------------------------------------------------
Photo                   0.0000         0.3248         0.4002         0.5270
Art Painting            0.3248         0.0000         0.3230         0.4883
Cartoon                 0.4002         0.3230         0.0000         0.4150
Sketch                  0.5270         0.4883         0.4150         0.0000

Distanza media dalle rispettive sorgenti:
Cartoon target: 0.3794
Sketch target:  0.4768
